# Поиск аномалий

Методы обнаружения аномалий, как следует из названия, позволяют находить необычные объекты в выборке. Но что такое "необычные" и совпадает ли это определение у разных методов?

Начнём с поиска аномалий в текстах: научимся отличать вопросы о программировании от текстов из 20newsgroups про религию.

Подготовьте данные: в обучающую выборку возьмите 20 тысяч текстов из датасета Stack Overflow, а тестовую выборку сформируйте из 10 тысяч текстов со Stack Overflow и 100 текстов из класса soc.religion.christian датасета 20newsgroups (очень пригодится функция `fetch_20newsgroups(categories=['soc.religion.christian'])`). Тексты про программирование будем считать обычными, а тексты про религию — аномальными.

In [1]:
import sklearn as sk
from sklearn.model_selection import train_test_split

In [2]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(
    categories=['soc.religion.christian'],
    shuffle=True,
    random_state=42,

)

texts = data.data[:100]


In [3]:
import pandas as pd
import numpy as np

df = pd.read_parquet("post_questions_test_000000000000.parquet")
df = df["body"].reset_index(drop = True)
x_train, x_test = train_test_split(df, train_size = 20000, test_size = 10000, random_state = 52, shuffle = True)
y_train = pd.Series([0] * len(x_train))
y_test = pd.Series([0] * len(x_test) + [1] * len(texts))
texts = pd.Series(texts)
x_test = pd.concat([x_test, texts])
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop = True)

perm = np.random.permutation(len(x_test))
x_test = x_test.iloc[perm]
y_test = y_test.iloc[perm]
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop = True)
print(x_test[10000:10100])

10000    <blockquote>\n  <p><strong>Possible Duplicate:...
10001    <p>i'm learning asp.net mvc \nwhen i add a mig...
10002    <p>the javascript to pass params:</p>\n\n<pre>...
10003    <p>I have an Angular 5 project, and it works f...
10004    <pre><code>DataTable dt = new DataTable();\ndt...
                               ...                        
10095    <p>I am trying to set up a code for Hazelcast ...
10096    <p>I have these three columns in UI. In dropdo...
10097    <p>We've just started using LINQ to SQL at wor...
10098    <p>I am trying to update firstname,lastname,im...
10099    <p>I am facing problem in registering new sql ...
Length: 100, dtype: object


**(1 балл)**

Проверьте качество выделения аномалий (pre и rec на тестовой выборке, если считать аномалии положительным классов, а обычные тексты — отрицательным) для IsolationForest. В качестве признаков используйте TF-IDF, где словарь и IDF строятся по обучающей выборке. Не забудьте подобрать гиперпараметры.

In [4]:
print(x_test)

0        <p>I have seen the Apple's example of Singleto...
1        <p>I have a declarative pipeline.\nIn this pip...
2        <p>Currently my team uses Visual Sourcesafe, a...
3        <p>I am new to Django. I want to add just a fe...
4        <p>In C++, one can define <code>bool operator&...
                               ...                        
10095    <p>I am trying to set up a code for Hazelcast ...
10096    <p>I have these three columns in UI. In dropdo...
10097    <p>We've just started using LINQ to SQL at wor...
10098    <p>I am trying to update firstname,lastname,im...
10099    <p>I am facing problem in registering new sql ...
Length: 10100, dtype: object


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score

vec = TfidfVectorizer(max_features=5000) 
x_train_v = vec.fit_transform(x_train)
x_test_v  = vec.transform(x_test)
iso = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination=0.01,
    random_state=42
)
iso.fit(x_train_v)
y_pred = iso.predict(x_test_v)




In [6]:
print(len(y_pred))
for i in range(5):
    if(y_pred[i] == -1):

        print(x_test[i][:20])


10100


In [7]:

y_pred = (y_pred == -1).astype(int)
# print(y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
print(pre, rec)

0.09482758620689655 0.11


**(5 баллов)**

Скорее всего, качество оказалось не на высоте. Разберитесь, в чём дело:
* посмотрите на тексты, которые выделяются как аномальные, а также на слова, соответствующие их ненулевым признакам
* изучите признаки аномальных текстов
* посмотрите на тексты из обучающей выборки, ближайшие к аномальным; действительно ли они похожи по признакам?

Сделайте выводы и придумайте, как избавиться от этих проблем. Предложите варианты двух типов: (1) в рамках этих же признаков (но которые, возможно, будут считаться по другим наборам данных) и методов и (2) без ограничений на изменения. Реализуйте эти варианты и проверьте их качество.

In [8]:
an_idx = np.where(y_pred == 1)[0]
print(len(y_pred))
for i in an_idx[:5]:
    
    print(x_test[i][0:300])
    pass


10100
<p>I tried taking screencapture from frame buffer and it works well for my layout views
I used the following code from <a href="https://stackoverflow.com/a/5651242/1206201">here</a>:</p>

<pre><code>String mPath = Environment.getExternalStorageDirectory().toString() + "/" + ACCUWX.IMAGE_APPEND;   


<h1>Background</h1>

<p>Trying to get into the spirit of TypeScript, I am writing fully typed signatures in my Components and Services, which extends to my custom validation functions for angular2 forms.</p>

<p>I know that <a href="https://www.typescriptlang.org/docs/handbook/functions.html" rel="n
<p>I an using <a href="https://www.npmjs.com/package/cordova-plugin-contacts" rel="nofollow">cordova-plugin-contacts</a> to pick a contact from contacts.
App is working fine on Android 5(Lollipop) and prior versions. But on Android 6(Marshmallow) app crashes when I pick a contact.</p>

<p>Here is my
<p>I am trying to implement an HTTP Server using Sockets. If the Client (For example a brow

In [9]:
feature_names = np.array(vec.get_feature_names_out())

def top_words(row, topn=10):
    arr = row.toarray().ravel()
    idx = np.argsort(arr)[-topn:]
    return feature_names[idx]

for i in an_idx[:5]:
    print(top_words(x_test_v[i]))

['views' 'printstacktrace' 'todo' 'camera' 'capture' 'view' 'catch'
 'layout' 'v1' 'bitmap']
['https' 'type' 'github' 'function' 'microsoft' 'issues' 'code' 'h1' 'li'
 'typescript']
['public' 'zygoteinit' 'final' 'at' 'contacts' 'contact' 'android' 'java'
 'cordova' 'activitythread']
['buffer' 'occurs' 'catch' 'method' 'at' 'socket' 'the' 'ioexception'
 'request' 'java']
['functions' 'structure' 'and' 'surface' 'to' 'event' 'of' 'integrate'
 'the' 'game']


In [10]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=3, metric='cosine')
nn.fit(x_train_v)

dist, ind = nn.kneighbors(x_test_v[an_idx][:5])

for i, idx in enumerate(an_idx[:5]):
    print("anom:")
    print(x_test.iloc[idx][:20])
    print("\nknn:")
    for j in ind[i]:
        print("-", x_train.iloc[j][:20])
    print("="*80)


anom:
<p>I tried taking sc

knn:
- <p>I need to copy a 
- <p>i have a huge mem
- <p>I have a Bitmap i
anom:
<h1>Background</h1>


knn:
- <p>In Java, you can 
- <p>Both epydoc and S
- <p>Valid value:</p>

anom:
<p>I an using <a hre

knn:
- <p>I'm trying to dow
- <p>I am getting erro
- <p>I am trying to ma
anom:
<p>I am trying to im

knn:
- <p>I am trying to wr
- <p>I am recently wor
- <p>So here's how it 
anom:
<p>Sorry for the rat

knn:
- <p>this is my first 
- <h3>Hello, is it pos
- <p>I am creating a b


### Эксперимент только с изменением датасета

In [11]:
import re
import pandas as pd

def sep_html(text):
    text = re.sub(r'(<[^>]+>)', r' \1 ', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()
    
x_train = x_train.apply(sep_html)
x_test = x_test.apply(sep_html)


# print(x_test)
# x_test

In [12]:
vec = TfidfVectorizer(
    max_features=5000
)
x_train_v = vec.fit_transform(x_train)
x_test_v = vec.transform(x_test)
iso = IsolationForest(
    n_estimators=100,
    max_samples="auto",
    contamination=0.01,
    random_state=42,
)
iso.fit(x_train_v)
y_pred = iso.predict(x_test_v)


y_pred = (y_pred == -1).astype(int)
# print(y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
print(pre, rec)

0.09482758620689655 0.11


### Эксперимент с любыми изменениями

In [13]:
import re
import html
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score
from sklearn.neighbors import NearestNeighbors


def getn(text):


    code_ptn = re.compile(r'(<pre.*?>.*?</pre>|<code.*?>.*?</code>|```[\s\S]*?```)', flags=re.I)
    html_tag_ptn = re.compile(r'(<[^>]+>)')
    email_header_ptn = re.compile(r'(?m)^(From|Subject|Reply-To|Organization|In article|Lines|Date):')

    quote_ptn = re.compile(r'(?m)^(>+)\s*')
    url_ptn = re.compile(r'https?://\S+|www\.\S+')
    email_addr_ptn = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')

    s = str(text)
    s = html.unescape(s)

    codes = code_ptn.findall(s)
    n_code_blocks = len(codes)
    s = code_ptn.sub(' __CODE_BLOCK__ ', s)

    tags = html_tag_ptn.findall(s)
    n_html_tags = len(tags)
    s = html_tag_ptn.sub(' __HTML_TAG__ ', s)

    n_email_headers = sum(1 for _ in email_header_ptn.finditer(s)) 
    s = email_header_ptn.sub(' __EMAIL_HEADER__ ', s)

    n_urls = len(url_ptn.findall(s))
    s = url_ptn.sub(' __URL__ ', s)

    n_emails = len(email_addr_ptn.findall(s)) 
    s = email_addr_ptn.sub(' __EMAIL_ADDR__ ', s)


    s = re.sub(r'\s+', ' ', s).strip()

    ft = {
        # 'n_code_blocks': n_code_blocks,
        # 'n_html_tags': -n_html_tags,
        'n_email_headers': n_email_headers,
        # 'n_urls': n_urls,
        'n_emails': n_emails,
    }

    return s, ft


In [14]:
def bf(x_train, x_test):
    alls = pd.concat([x_train, x_test], ignore_index=True)

    clt = []
    ftrtr = []

    for t in alls:
        c, f = getn(t)
        clt.append(c)
        ftrtr.append(f)

    df_features = pd.DataFrame(ftrtr).fillna(0)

    n_train = len(x_train)

    clean_train = pd.Series(clt[:n_train])
    clean_test = pd.Series(clt[n_train:])

    df_train = df_features.iloc[:n_train].reset_index(drop=True)
    df_test = df_features.iloc[n_train:].reset_index(drop=True)

    return clean_train, clean_test, df_train, df_test


In [15]:

from sklearn.preprocessing import StandardScaler
from scipy import sparse

scaler = StandardScaler()


def bvct(clean_train, clean_test, df_train, df_test):
    vec = TfidfVectorizer(
        max_features=3000,
        stop_words='english',
        min_df=3,
        ngram_range=(1,2),
        sublinear_tf=True
    )

    X_tfidf_train = vec.fit_transform(clean_train)
    X_tfidf_test = vec.transform(clean_test)

    scaler = StandardScaler()
    X_num_train = scaler.fit_transform(df_train.values)
    X_num_test = scaler.transform(df_test.values)
    coef = 100000000
    X_num_train = scaler.fit_transform(df_train.values * coef)
    X_num_test  = scaler.transform(df_test.values * coef)
    X_train = sparse.hstack([X_tfidf_train, sparse.csr_matrix(X_num_train)])
    X_test = sparse.hstack([X_tfidf_test, sparse.csr_matrix(X_num_test)])
    
    X_train = sparse.hstack([sparse.csr_matrix(X_num_train)])
    X_test = sparse.hstack([sparse.csr_matrix(X_num_test)])

    return X_train, X_test, vec


In [16]:
def run(X_train, X_test, y_test):
    iso = IsolationForest(
        n_estimators=200,
        contamination=0.00990099009900990099009900990099,
        random_state=42
    )

    iso.fit(X_train)
    scores = iso.decision_function(X_test)  # array float
    # threshold_99 = np.percentile(scores, 100-0.990099009900990099009900990099)

    # y_pred = (scores >= threshold_99).astype(int)  # 1 = аномалия
    y_pred = (scores < 0).astype(int)  # 1 = аномалия

    


    pre = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    print("Precision:", pre)
    print("Recall:", rec)

    return y_pred


In [17]:
def showan(y_pred, clean_test, df_features_test, n=10):
    idx = np.where(y_pred == 1)[0]

    print("Tot:", len(idx))

    for i in idx[:5]:
        print("\n" + "-"*70)
        print("Text:")
        print(clean_test.iloc[i][:400])
        print("\nNumeric ft:")
        print(df_features_test.iloc[i].to_dict())


In [18]:
clean_train, clean_test, df_train, df_test = bf(x_train, x_test)

X_train, X_test, vec = bvct(
    clean_train, clean_test,
    df_train, df_test
)
# df_train['n_email_headers'] *= 10000000
# df_test['n_email_headers']  *= 10000000

y_pred = run(X_train, X_test, y_test)

showan(y_pred, clean_test, df_test, n=110)


Precision: 0.8448275862068966
Recall: 0.98
Tot: 116

----------------------------------------------------------------------
Text:
__EMAIL_HEADER__ __EMAIL_ADDR__ (Steve Lang) Subject: Re: The arrogance of Christians Organization: Nottingham University Lines: 60 In article __HTML_TAG__ , you wrote: > The genius of science is that it discovered that enormous progress > in knowledge could be made by isolating the study of physical > interactions for the more general areas of study and proceeding > not by logical argument but by

Numeric ft:
{'n_email_headers': 1, 'n_emails': 1}

----------------------------------------------------------------------
Text:
__EMAIL_HEADER__ __EMAIL_ADDR__ (Mark Baker) Subject: Re: The arrogance of Christians Reply-To: __EMAIL_ADDR__ (Mark Baker) Organization: The National Capital Freenet Lines: 22 In a previous article, __EMAIL_ADDR__ (Melinda . Hsu) says: > >Well the argument usually stops right there. In the end, >aren't we all just kids, groping for the t

Подготовьте выборку: удалите столбцы `['id', 'date', 'price', 'zipcode']`, сформируйте обучающую и тестовую выборки по 10 тысяч домов.

Добавьте в тестовую выборку 10 новых объектов, в каждом из которых испорчен ровно один признак — например, это может быть дом из другого полушария, из далёкого прошлого или будущего, с площадью в целый штат или с таким числом этажей, что самолётам неплохо бы его облетать стороной.

Посмотрим на методы обнаружения аномалий на более простых данных — уж на табличном датасете с 19 признаками всё должно работать как надо!

Скачайте данные о стоимости домов: https://www.kaggle.com/harlfoxem/housesalesprediction/data

In [19]:
#code here

**Задание 9. (2 балла)**

Примените IsolationForest для поиска аномалий в этих данных, запишите их качество (как и раньше, это pre и rec). Проведите исследование:

Нарисуйте распределения всех признаков и обозначьте на этих распределениях объекты, которые признаны аномальными.

In [20]:
#code here